## c21 - GNN Surrogate Model Training
Trains `TrussEdgeSafetyGNN` to predict per-member structural safety (UC > 1.0) from node and edge features. Sections: preprocessing → training → evaluation → export.

## 1. Preprocessing
Load edge and node CSVs, build PyG graph objects, and split into train/val/test.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "workflows"))
from c21_surrogate_training import run_preprocessing

# ── Parameters ───────────────────────────────────────────────────────────────
EDGE_CSV        = "v6_edge_C14_S19999_D20260516"
NODE_CSV        = "v6_node_C12_S19999_D20260516"
BIDIRECTIONAL   = False
BATCH_SIZE      = 64
DATA_INSPECTION = False

HIDDEN_DIM = 64   # ~1.1M params with weight_decay + dropout
NUM_LAYERS = 4    # 4 hops needed to see full load paths across the truss
DROPOUT_P  = 0.3

# ── Run ───────────────────────────────────────────────────────────────────────
pre = run_preprocessing(
    EDGE_CSV, NODE_CSV,
    bidirectional   = BIDIRECTIONAL,
    batch_size      = BATCH_SIZE,
    data_inspection = DATA_INSPECTION,
    hidden_dim      = HIDDEN_DIM,
    num_layers      = NUM_LAYERS,
    dropout_p       = DROPOUT_P,
)

# 2. Training
Run the training loop with early stopping and learning-rate scheduling. Adjust `EPOCHS`, `PATIENCE`, and `POS_WEIGHT` as needed.

In [ ]:
from c21_surrogate_training import run_training

# ── Parameters ───────────────────────────────────────────────────────────────
EPOCHS            = 200
LR                = 1e-4
PATIENCE          = 60
LR_FACTOR         = 0.5
LR_PATIENCE       = 15
LR_MIN            = 1e-6
GRAD_CLIP         = 1.0
WEIGHT_DECAY      = 1e-3
DEFAULT_THRESHOLD = 0.35
MIN_PRECISION     = 0.40
POS_WEIGHT        = 2.5   # lower than auto ~4.1: fewer FP, higher precision at cost of recall

# ── Run ───────────────────────────────────────────────────────────────────────
train_out = run_training(
    pre,
    epochs            = EPOCHS,
    lr                = LR,
    patience          = PATIENCE,
    lr_factor         = LR_FACTOR,
    lr_patience       = LR_PATIENCE,
    lr_min            = LR_MIN,
    grad_clip         = GRAD_CLIP,
    weight_decay      = WEIGHT_DECAY,
    default_threshold = DEFAULT_THRESHOLD,
    min_precision     = MIN_PRECISION,
    pos_weight        = POS_WEIGHT,
)

# 3. Evaluation
Evaluate the trained model on the held-out test set. Reports precision, recall, F1, ROC-AUC, and per-design feasibility accuracy.

In [ ]:
from c21_surrogate_training import run_evaluation

# ── Run ───────────────────────────────────────────────────────────────────────
eval_out = run_evaluation(train_out, pre)


# 4. Export
Save the trained model weights, feature scaler, and metadata to disk. The output prefix is used by the optimizer to load the surrogate bundle.

In [ ]:
from c21_surrogate_training import run_export

# ── Run ───────────────────────────────────────────────────────────────────────
export_out = run_export(pre, train_out, eval_out)
print("Saved to:", export_out["models_dir"])
print("Files:")
for f in export_out["all_files"]:
    print(" ", f)
